# From the Mini Model to MESSAGEix

**ENGG562/ENV462 · Climate Change Governance: Science, Data & Models · LUMS · Fall 2026**

**Module 2: Scenarios & Integrated Assessment** · Coding exercise 07 · Mini integrated assessment model series, notebook 5 of 5 · Requires coding exercises 04–06

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LUMS-WIT/climate-futures/blob/main/content-src/coding/07-from-mini-iam-to-messageix.ipynb)

Notebooks 04 to 06 used `miniiam`, a teaching-scale copy of MESSAGEix. This notebook runs the **same code** on the real framework, the open-source package [`message_ix`](https://docs.messageix.org) developed at IIASA, which is used for the global MESSAGEix-GLOBIOM model of the IPCC scenarios and for many national energy and climate studies.

The notebook is written so that every cell works with either package:

- **On a computer with `message_ix` installed** (Part 1), the cells use the real model, solved with GAMS.
- **In Google Colab**, where `message_ix` and GAMS are not available, the same cells fall back to `miniiam`. The code is identical; only the solver behind it changes.

| Notebook | Content |
|---|---|
| 03 | Linear optimisation, shadow prices, and a small capacity expansion model |
| 04 | Westeros: the MESSAGE formulation on a small example |
| 05 | World: the same formulation applied to a global energy system |
| 06 | MESSAGE-MACRO: linking the energy model to a macro-economic growth model |
| **07 (this notebook)** | **From the mini model to the real `message_ix` package** |

## Learning objectives

On completing this notebook, students should be able to:

1. Install `message_ix` and create a scenario on a local database (the *ix modeling platform*, ixmp).
2. Run the Westeros scenarios of notebooks 04 and 06 on MESSAGEix and compare the results with the mini model.
3. Explain the differences between the mini model and MESSAGEix and their effect on results.
4. Report model results in the IAMC format used by the IPCC scenario database (coding exercise 02).

## Structure of the notebook

| Part | Content |
|---|---|
| 1 | Installing MESSAGEix |
| 2 | What changes between `miniiam` and `message_ix` |
| 3 | Westeros on MESSAGEix: baseline, emission bound, and tax |
| 4 | MESSAGE-MACRO on MESSAGEix |
| 5 | The World model on MESSAGEix |
| 6 | Reporting results in the IAMC format |
| 7 | Where to go next |

---
# Part 1 · Installing MESSAGEix

MESSAGEix has three components:

1. the Python packages `message_ix` and `ixmp`, which provide the `Scenario` interface used throughout this series;
2. **GAMS**, the modelling system in which the MESSAGE and MACRO equations are written and which calls the solver; and
3. a database in which scenarios are stored (by default a local file database that requires **Java**).

The installation steps for each operating system are given in the [MESSAGEix installation guide](https://docs.messageix.org/en/stable/install.html). In short:

- install GAMS from [gams.com](https://www.gams.com/download/) and add it to the system `PATH`; the free licences available from GAMS are sufficient for Westeros;
- install a Java runtime (JDK);
- install the Python packages, for example with `pip install message_ix` or `conda install -c conda-forge message-ix`;
- check the installation by running `message-ix show-versions` in a terminal.

The cell below detects which package is available. In Colab, it downloads `miniiam` and uses it instead.

In [ ]:
import sys

if "google.colab" in sys.modules:
    !pip install --quiet pyomo highspy
    !wget --quiet --timestamping https://raw.githubusercontent.com/LUMS-WIT/climate-futures/main/content-src/coding/miniiam.py

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import miniiam as mi

try:
    import ixmp
    import message_ix
    from message_ix import make_df
    USE_MESSAGE_IX = True
    # An in-memory database: nothing is written to disk, and it is discarded on exit
    mp = ixmp.Platform(backend="jdbc", driver="hsqldb", url="jdbc:hsqldb:mem:course")
    print("Using message_ix", message_ix.__version__)
except Exception as err:
    from miniiam import make_df
    USE_MESSAGE_IX = False
    mp = None
    print("message_ix is not available here; using the mini model instead.", f"({type(err).__name__})")


def new_scenario(model, scenario):
    """Create an empty scenario in whichever package is available."""
    if USE_MESSAGE_IX:
        return message_ix.Scenario(mp, model=model, scenario=scenario, version="new")
    return mi.Scenario(model=model, scenario=scenario)


def objective(s):
    """OBJ is a number in miniiam and a dict with keys 'lvl' and 'mrg' in message_ix."""
    obj = s.var("OBJ")
    return obj["lvl"] if isinstance(obj, dict) else obj

In [ ]:
def check(description, condition):
    """Self-check helper: prints a tick or a cross. It never stops the notebook."""
    try:
        passed = bool(condition())
    except Exception:
        passed = False
    print(("✅ " if passed else "❌ ") + description)

---
# Part 2 · What changes between `miniiam` and `message_ix`

The commands used in notebooks 04–06 were chosen to be those of `message_ix`. Four differences remain:

| Topic | `miniiam` | `message_ix` |
|---|---|---|
| Storage | Scenarios live in memory | Scenarios are stored in a database (`ixmp.Platform`) with version numbers |
| Editing | Data can be added at any time | A scenario must be **checked out** before editing and **committed** afterwards; a solved scenario must be cloned with `keep_solution=False` before it is edited |
| Units | Not checked | Every unit must be registered on the platform (`mp.add_unit`) |
| Solver | Pyomo with HiGHS (MESSAGE), SciPy (MACRO) | GAMS with CPLEX or another LP solver (MESSAGE), CONOPT or another NLP solver (MACRO) |
| Results | Activity by active year | Activity by vintage *and* active year, mode, and time slice |

The mini model accepts `check_out()` and `commit()` and does nothing with them, so the code below runs unchanged on both. The data builders `mi.build_westeros()` and `mi.build_world()` register their units automatically when they receive a `message_ix` scenario.

The formulation also differs in detail: MESSAGEix tracks efficiencies and costs by vintage, supports several modes of operation and sub-annual time slices, storage, trade between regions, land-use emulators, and many more constraint types. The mini model contains the core of the formulation only.

---
# Part 3 · Westeros on MESSAGEix

The function `mi.build_westeros()` contains exactly the steps written by hand in Part 2 of notebook 04. Here it fills a `message_ix` scenario (or, in Colab, a mini-model scenario).

In [ ]:
base = new_scenario("Westeros Electrified", "baseline")
mi.build_westeros(base, emissions=False)
base.commit("Westeros baseline, as in notebook 04")
base.solve(quiet=True)
print(f"Objective: {objective(base):,.0f} million USD")

In [ ]:
def activity(s, techs=("coal_ppl", "wind_ppl")):
    act = s.var("ACT")
    return act[act.technology.isin(techs)].groupby(["year_act", "technology"])["lvl"].sum().unstack()

# The mini model, for comparison
mini = mi.build_westeros(mi.Scenario("Westeros Electrified", "baseline (mini)"), emissions=False)
mini.solve()

comparison = pd.concat({"this run": activity(base), "miniiam": activity(mini)}, axis=1)
print(f"Objective: this run {objective(base):,.0f} | miniiam {objective(mini):,.0f} million USD")
comparison.round(1)

**💬 Question.** If this notebook runs on `message_ix`, compare its activity and objective with the mini model. Which results agree closely, which differ, and what explains the difference? (If it runs on the mini model in Colab, answer with reference to the trainer's demonstration or a classmate's installation.)

*Answer:* *(write the answer here)*

## Emission bound and emission tax

The steps of notebook 04, Parts 5 and 6. Note the pattern for editing a solved scenario in `message_ix`: **clone without the solution, check out, edit, commit, solve**.

In [ ]:
country = "Westeros"
scen_bd = base.clone(scenario="emission_bound", keep_solution=False)
scen_bd.check_out()
scen_bd.add_set("emission", "CO2")
scen_bd.add_cat("emission", "GHG", "CO2")

years = scen_bd.vintage_and_active_years((country, "coal_ppl"), in_horizon=False)
scen_bd.add_par("emission_factor", make_df(
    "emission_factor", node_loc=country, year_vtg=years["year_vtg"], year_act=years["year_act"],
    mode="standard", unit="tCO2/kWa", technology="coal_ppl", emission="CO2", value=7.4))
scen_bd.add_par("bound_emission", make_df(
    "bound_emission", node=country, type_emission="GHG", type_tec="all",
    type_year="cumulative", value=500.0, unit="MtCO2"))
scen_bd.commit("cumulative emission bound of 500 MtCO2 per year on average")
scen_bd.solve(quiet=True)

prices = scen_bd.var("PRICE_EMISSION").groupby("year")["lvl"].sum()
prices.to_frame("carbon price (USD/t CO2)")

### ✏️ Exercise 1: The tax scenario on MESSAGEix

Create `scen_tax` from `base` with an emission tax equal to `prices` in each year, following the editing pattern above. In `message_ix`, the model years must also be added to the set `type_year` before they can be used in `tax_emission` (in the mini model, `add_set` simply records them).

In [ ]:
scen_tax = base.clone(scenario="carbon_tax", keep_solution=False)
scen_tax.check_out()
scen_tax.add_set("emission", "CO2")
scen_tax.add_cat("emission", "GHG", "CO2")
scen_tax.add_par("emission_factor", scen_bd.par("emission_factor"))
scen_tax.add_set("type_year", [700, 710, 720])
scen_tax.add_par("tax_emission", make_df(
    "tax_emission", node=country, type_emission="GHG", type_tec="all",
    type_year=[700, 710, 720], value=___, unit="USD/tCO2"))
scen_tax.commit("carbon tax equal to the shadow price of the bound")
scen_tax.solve(quiet=True)

em = pd.DataFrame({"bound": scen_bd.var("EMISS").groupby("year")["lvl"].sum(),
                   "tax": scen_tax.var("EMISS").groupby("year")["lvl"].sum()})
em.round(1)

In [ ]:
check("the tax keeps average emissions within the bound", lambda: em["tax"].mean() <= 500 + 1)

---
# Part 4 · MESSAGE-MACRO on MESSAGEix

`add_macro()` takes the same input data as in notebook 06 and returns a calibrated clone. In `message_ix`, calibration runs a fixed number of iterations in GAMS and then checks convergence by solving MESSAGE-MACRO once.

Two details of the input data are worth knowing, because they are easy to get wrong: `gdp_calibrate` and `cost_ref` are read in **billion** USD (the MESSAGEix test data for Westeros label them "T$", but the model treats them as G$), and `gdp_calibrate` must contain at least two periods before the first model year, so that the base-year growth rate can be computed.

In [ ]:
macro = base.add_macro(mi.westeros_macro_data())

bound_mm = macro.clone(scenario="emission_bound (MESSAGE-MACRO)", keep_solution=False)
bound_mm.check_out()
bound_mm.add_set("emission", "CO2")
bound_mm.add_cat("emission", "GHG", "CO2")
bound_mm.add_par("emission_factor", scen_bd.par("emission_factor"))
bound_mm.add_par("bound_emission", scen_bd.par("bound_emission"))
bound_mm.commit("emission bound with MESSAGE-MACRO")
bound_mm.solve(model="MESSAGE-MACRO", quiet=True)

gdp = pd.DataFrame({"baseline": macro.var("GDP").groupby("year")["lvl"].sum(),
                    "emission bound": bound_mm.var("GDP").groupby("year")["lvl"].sum()})
gdp["loss (%)"] = 100 * (1 - gdp["emission bound"] / gdp["baseline"])
gdp.round(2)

**💬 Question.** Compare the GDP loss and the demand reduction with those of notebook 06. If they differ, which features of the two implementations could explain the difference?

*Answer:* *(write the answer here)*

---
# Part 5 · The World model on MESSAGEix

`mi.build_world()` also accepts a `message_ix` scenario. With 29 technologies and eight periods, the World model is larger than Westeros; with a size-limited free GAMS licence, it may fail to solve with a licence error, in which case an academic licence is needed.

In [ ]:
world = new_scenario("World", "baseline")
mi.build_world(world)
world.commit("World baseline, as in notebook 05")
try:
    world.solve(quiet=True)
    print(f"Objective: {objective(world) / 1e6:.1f} trillion USD")
    world_ok = True
except Exception as err:
    print("The World model could not be solved here:", str(err)[:300])
    world_ok = False

In [ ]:
if world_ok:
    w2c = world.clone(scenario="2C (1150 Gt)", keep_solution=False)
    w2c.check_out()
    w2c.add_par("bound_emission", make_df("bound_emission", node="World", type_emission="GHG", type_tec="all",
                                          type_year="cumulative", value=1150 * 1000 / 80, unit="MtCO2"))
    w2c.commit("2 degree budget")
    w2c.solve(quiet=True)
    print(w2c.var("PRICE_EMISSION").groupby("year")["lvl"].sum().round(1).to_dict())

---
# Part 6 · Reporting results in the IAMC format

The IPCC scenario database of coding exercise 02 stores every model's results in the IAMC format (model, scenario, region, variable, unit, years). MESSAGEix has a full reporting system for this purpose (`message_ix.report`); the mini model has a small function, `mi.to_iamc()`, that reports emissions, electricity generation by source, the carbon price, and GDP under the IAMC variable names.

With results in this format, the mini model can be placed directly beside the AR6 scenarios, using the `pyam` package of coding exercise 02.

In [ ]:
w = mi.build_world(mi.Scenario("mini World", "baseline"))
w.solve()
w2 = w.clone(scenario="2C (1150 Gt)")
w2.add_par("bound_emission", make_df("bound_emission", node="World", type_emission="GHG", type_tec="all",
                                     type_year="cumulative", value=1150 * 1000 / 80, unit="MtCO2"))
w2.solve()

iamc = pd.concat([mi.to_iamc(w), mi.to_iamc(w2)])
iamc.pivot_table(index=["scenario", "variable"], columns="year", values="value").round(1).head(14)

### ✏️ Exercise 2: The mini model beside the AR6 scenarios · ★ Stretch

Install `pyam` (as in coding exercise 02), convert `iamc` to a `pyam.IamDataFrame`, and query the AR6 database for global CO₂ emissions (`Emissions|CO2`) in categories C3 and C7. Plot the AR6 ranges and the two mini-model scenarios on one figure (both are in Mt CO₂ per year). This exercise needs an internet connection to the IIASA database; the troubleshooting note in Part 0 of coding exercise 02 applies.

In [ ]:
if "google.colab" in sys.modules:
    !pip install --quiet pyam-iamc
import pyam

mini_df = pyam.IamDataFrame(iamc)   # the mini model as a pyam object, for any further pyam analysis
conn = pyam.iiasa.Connection("ar6-public")
ar6 = conn.query(variable="Emissions|CO2", region="World", meta=["Category"])
ar6 = ar6.filter(Category=["C3", "C7"])

fig, ax = plt.subplots()
ar6.filter(year=range(2020, 2101)).plot(ax=ax, color="Category", fill_between=True)
for name, g in iamc[iamc["variable"] == ___].groupby("scenario"):
    ax.plot(g["year"], g["value"], linewidth=3, label=f"mini model: {name}")
ax.legend(fontsize=8)
plt.show()

---
# Part 7 · Where to go next

The mini model has the structure of MESSAGE. The models used in research and policy add detail in every dimension, but the concepts of this series carry over unchanged: a reference energy system, least-cost optimisation with growth constraints, shadow prices, emission bounds and taxes, and the MACRO linkage.

- **The official tutorials.** The [MESSAGEix tutorials](https://github.com/iiasa/message_ix/tree/main/tutorial) continue from where notebook 04 stopped: renewable resource potentials, fossil resources, firm capacity, flexibility, seasonality, trade between regions, and a model of the Austrian energy system.
- **MESSAGEix-GLOBIOM and `message-ix-models`.** The global model used for the IPCC, with 12 world regions and a link to the land-use model GLOBIOM, is built with the open-source package [`message-ix-models`](https://docs.messageix.org/projects/models/en/latest/), which also supports building national models.
- **The course project.** A national MESSAGEix model, or the mini model extended with Pakistan-specific data, is one of the methods suggested on the course [Project page](https://lums-wit.github.io/climate-futures/project.html).

**💬 Final question.** Name one question about Pakistan's energy or climate policy that a MESSAGE-type model could help to answer, and one that it could not. Explain both in two or three sentences.

*Answer:* *(write the answer here)*

---
# References

- Huppmann, D., Gidden, M., Fricko, O., Kolp, P., Orthofer, C., Pimmer, M., et al. (2019). The MESSAGEix Integrated Assessment Model and the ix modeling platform (ixmp). *Environmental Modelling & Software*, 112, 143–156. [doi:10.1016/j.envsoft.2018.11.012](https://doi.org/10.1016/j.envsoft.2018.11.012)
- [MESSAGEix documentation](https://docs.messageix.org), including the installation guide, the mathematical formulation, and the MACRO calibration guide.
- Messner, S., and Schrattenholzer, L. (2000). MESSAGE–MACRO: linking an energy supply model with a macroeconomic module and solving it iteratively. *Energy*, 25(3), 267–282. [doi:10.1016/S0360-5442(99)00063-8](https://doi.org/10.1016/S0360-5442(99)00063-8)
- Huppmann, D., Gidden, M. J., Nicholls, Z., Hörsch, J., Lamboll, R., Kishimoto, P. N., et al. (2021). pyam: Analysis and visualisation of integrated assessment and macro-energy scenarios. *Open Research Europe*, 1:74. [doi:10.12688/openreseurope.13633.2](https://doi.org/10.12688/openreseurope.13633.2)